# Notebook 2: Financial Analysis

**Project:** Automated Equity Research Platform

This notebook runs independently from the other notebooks.

## Standalone setup

This notebook is designed to run by itself. It can use a local clone, a Google Drive folder, or a GitHub repository.

Before publishing, replace `YOUR_USERNAME` in `REPO_URL` with your actual GitHub username.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_NAME = "Automated_Equity_Research_Platform"
REPO_URL = "https://github.com/YOUR_USERNAME/Automated_Equity_Research_Platform.git"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

candidate_roots = [
    Path.cwd(),
    Path("/content") / PROJECT_NAME,
    Path("/content/drive/MyDrive") / PROJECT_NAME,
    Path("/content/drive/MyDrive/AI_Equity_Research_Platform"),
]

PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "pyproject.toml").exists()),
    None,
)

if PROJECT_ROOT is None:
    if "YOUR_USERNAME" in REPO_URL:
        raise RuntimeError(
            "Project files were not found. Upload the full project folder to Google Drive "
            "or replace YOUR_USERNAME in REPO_URL with your real GitHub username."
        )
    target = Path("/content") / PROJECT_NAME
    if not target.exists():
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
    PROJECT_ROOT = target

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .

In [ ]:
import os

try:
    from google.colab import userdata
    for secret_name in ("SEC_USER_AGENT", "FRED_API_KEY"):
        try:
            value = userdata.get(secret_name)
            if value:
                os.environ[secret_name] = value
        except Exception:
            pass
except Exception:
    pass

print("SEC enabled:", bool(os.getenv("SEC_USER_AGENT")))
print("FRED enabled:", bool(os.getenv("FRED_API_KEY")))

Analyze financial statements, ratios, business quality, and historical statement trends.

In [ ]:
from equity_research.data import YahooFinanceClient
from equity_research.fundamentals import FundamentalAnalyzer
from equity_research.visualization import ratio_chart
import pandas as pd
import plotly.graph_objects as go

TICKER = input("Ticker: ").strip().upper() or "AAPL"

yahoo = YahooFinanceClient()
bundle = yahoo.bundle(TICKER, start="2016-01-01")
snapshot = FundamentalAnalyzer().analyze(bundle)

print(f"{TICKER} quality score: {snapshot.quality_score:.1f}/10")
display(pd.Series(snapshot.values, name="Value").to_frame())
display(snapshot.ratio_frame())
ratio_chart(snapshot.ratios).show()

In [ ]:
def statement_row(statement, candidates):
    normalized = {str(index).replace(" ", "").lower(): index for index in statement.index}
    for candidate in candidates:
        key = candidate.replace(" ", "").lower()
        if key in normalized:
            return pd.to_numeric(statement.loc[normalized[key]], errors="coerce").sort_index()
    return pd.Series(dtype=float)

series_list = [
    (statement_row(bundle.income_statement, ["TotalRevenue", "OperatingRevenue"]), "Revenue"),
    (statement_row(bundle.income_statement, ["OperatingIncome"]), "Operating Income"),
    (statement_row(bundle.income_statement, ["NetIncome", "NetIncomeCommonStockholders"]), "Net Income"),
    (statement_row(bundle.cash_flow, ["FreeCashFlow"]), "Free Cash Flow"),
]

fig = go.Figure()
for series, label in series_list:
    if not series.empty:
        fig.add_trace(go.Bar(x=series.index, y=series.values, name=label))
fig.update_layout(title=f"{TICKER} Financial Trends", barmode="group", template="plotly_white", height=520)
fig.show()